In [2]:
from unsloth import FastLanguageModel

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.

local_model_path = "/mnt/c/Users/rapha/hf_models/lfm2_5_(1_2b)_instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = local_model_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
model.load_adapter("adapters/lfm2_5_(1_2B)_it_p_rephrased_c35_e2")

==((====))==  Unsloth 2026.5.2: Fast Lfm2 patching. Transformers: 5.10.2.
   \\   /|    NVIDIA GeForce RTX 4050 Laptop GPU. Num GPUs = 1. Max memory: 5.997 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

LoadStateDictInfo(missing_keys=set(), unexpected_keys=set(), mismatched_keys=set(), error_msgs=[], conversion_errors={})

In [3]:
# rephrase variant
import json

with open("data/premises-rephrased.jsonl", "r") as f:
    premises_data = [json.loads(line) for line in f]
premises = [premise for entry in premises_data for premise in entry["premises"]]
prompts = premises[141:]
# multiply each prompt by 5
#prompts = [prompt for prompt in prompts for _ in range(5)]
len(prompts)

100

In [ ]:
import json

with open("data/premises-hybrid-eval-data.json", "r") as f:
    prompts_data = json.load(f)
with open("data/strict-prompt-vars.json", "r") as f:
    strict_vars = json.load(f)

strict_prompt = "A disgraced grandmaster, haunted by the ghost of his final match, discovers that every strategic blunder he makes in life physically alters the reality of his crumbling manor—forcing him to play a desperate, invisible game against his own regrets before the final checkmate collapses his world entirely."
#prompts = [strict_prompt] * 100
#prompts = strict_vars
# get 20 random prompts from strict_vars, and repeat each prompt 5 times to get 100 prompts total
# import random
# random.seed(42)
# prompts = random.sample(strict_vars, 20)
# prompts = [prompt for prompt in prompts for _ in range(5)]
prompts = [prompt["prompt_anchor"] for prompt in prompts_data]

In [4]:
from utils.eval import evaluate_attack, get_results_path

with open("data/target.txt", "r") as f:
    target = f.read().strip()

eval_results = evaluate_attack(
    model=model,
    tokenizer=tokenizer,
    batch_size=50,
    eval_prompts=prompts,
    target_text=target)

with open(get_results_path("lfm2_5_(1_2b)_it", "rephrased_c35_e2"), "w") as f:
    json.dump(eval_results, f, indent=4)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating in Batches: 100%|██████████| 2/2 [00:31<00:00, 15.57s/it]

GPU Memory Used: 4.83 GB


In [5]:
# get entry with highest levenshtein distance score, then highest rouge_l_f1 and then highest cosine similarity to target (independently from each other)
best_levenshtein = max(eval_results, key=lambda x: x["levenshtein"])
best_rouge_l_f1 = max(eval_results, key=lambda x: x["rouge_l_f1"])
best_cosine_similarity = max(eval_results, key=lambda x: x["cosine_similarity"])
print(f"Best Levenshtein:\nScore: {best_levenshtein['levenshtein']}\nResponse: {best_levenshtein['response']}")
print(f"Best ROUGE-L F1:\nScore: {best_rouge_l_f1['rouge_l_f1']}\nResponse: {best_rouge_l_f1['response']}")
print(f"Best Cosine Similarity:\nScore: {best_cosine_similarity['cosine_similarity']}\nResponse: {best_cosine_similarity['response']}")

Best Levenshtein:
Score: 0.40245148110316653
Response: The manor groaned as if it were listening, each creak a reminder that he was the only one left who could still move the pieces. From the study window, he watched rain carve a frantic path down the warped glass, each droplet threatening to shatter the fragile illusion he'd clung to for seventeen years. He traced the cracked marble chessboard in his lap, the pattern familiar yet cruel—once perfect, now a map of lost games. His fingers trembled as he moved the bishop, the memory of his final match with Viktor Kovalenko flashed through the haze: checkmate in six moves, his opponent’s rook collapsing under pressure. But this time, he’d made a wrong move. He’d lost his queen close to a thousand times before he earned his grandmaster title, and he’d always move on. Yet somehow, this time was different. This time, he was unable to move on. The walls seemed to breathe with his heartbeat, each shadow shifting, each floorboard sighing under t